In [0]:
# --- Configuration: USING THE CORRECT 'landing' CONTAINER ---
CONTAINER_PATH = "abfss://landing@projectbigdata.dfs.core.windows.net"
RAW_TWEET_FILE_PATH = f"{CONTAINER_PATH}/stocktweet/stocktweet.csv"
TARGET_TWEET_TABLE = "project_raw.raw_tweets" # Saving in the project_raw schema

# 1. Create a Schema (Database) for raw data in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS project_raw;")

# 2. Read the raw CSV data (using multiLine and escape options for the dirty tweet file)
print(f"Reading raw tweet data from: {RAW_TWEET_FILE_PATH}")
df_tweets = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .option("quote", '"') \
    .option("multiLine", "true") \
    .load(RAW_TWEET_FILE_PATH)

# 3. Write the data to a Delta Lake table
df_tweets.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TWEET_TABLE)

print(f"✅ Successfully created Delta table: {TARGET_TWEET_TABLE}")
spark.sql(f"SELECT COUNT(*) AS total_tweets, min(date) as min_date, max(date) as max_date FROM {TARGET_TWEET_TABLE}").show()

Reading raw tweet data from: abfss://landing@projectbigdata.dfs.core.windows.net/stocktweet/stocktweet.csv
✅ Successfully created Delta table: project_raw.raw_tweets
+------------+----------+----------+
|total_tweets|  min_date|  max_date|
+------------+----------+----------+
|       10000|01/01/2020|31/12/2020|
+------------+----------+----------+



In [0]:
import re

def clean_column_name(col):
    """Replaces spaces with underscores and removes invalid characters."""
    # Replace spaces with underscore
    cleaned_col = col.replace(" ", "_")
    # Remove any other non-alphanumeric characters (like the period in 'Adj. Close' if present)
    cleaned_col = re.sub(r'[^\w]', '', cleaned_col)
    return cleaned_col

In [0]:
# --- Configuration: USING THE CORRECT 'landing' CONTAINER ---
CONTAINER_PATH = "abfss://landing@projectbigdata.dfs.core.windows.net"
RAW_PRICE_FOLDER_PATH = f"{CONTAINER_PATH}/stockprice/" 
TARGET_PRICE_TABLE = "project_raw.raw_prices"

# 1. Read all price CSV files from the folder
print(f"Reading all price CSVs from folder: {RAW_PRICE_FOLDER_PATH}")
df_prices = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(RAW_PRICE_FOLDER_PATH)

# --- NEW STEP: Rename Columns to be Delta-compatible ---
# Get the list of current column names
existing_cols = df_prices.columns
# Create a dictionary of old_name: new_name mappings
col_map = {col: clean_column_name(col) for col in existing_cols}

# Apply the renames to the DataFrame
for old_name, new_name in col_map.items():
    df_prices = df_prices.withColumnRenamed(old_name, new_name)

# 2. Write the data to a Delta Lake table
df_prices.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_PRICE_TABLE)

print(f"✅ Successfully created Delta table: {TARGET_PRICE_TABLE}")
spark.sql(f"SELECT * FROM {TARGET_PRICE_TABLE} LIMIT 5").show()

Reading all price CSVs from folder: abfss://landing@projectbigdata.dfs.core.windows.net/stockprice/
✅ Successfully created Delta table: project_raw.raw_prices
+----------+-----------------+-----------------+-----------------+-----------------+-----------------+----------+
|      Date|             Open|             High|              Low|            Close|        Adj_Close|    Volume|
+----------+-----------------+-----------------+-----------------+-----------------+-----------------+----------+
|2019-12-31|3215.179931640625|3231.719970703125|3212.030029296875|3230.780029296875|3230.780029296875|2894760000|
|2020-01-02|   3244.669921875|3258.139892578125|3235.530029296875| 3257.85009765625| 3257.85009765625|3459930000|
|2020-01-03|3226.360107421875| 3246.14990234375|3222.340087890625| 3234.85009765625| 3234.85009765625|3484700000|
|2020-01-06|3217.550048828125|3246.840087890625|3214.639892578125|3246.280029296875|3246.280029296875|3702460000|
|2020-01-07|3241.860107421875|3244.90991210